# GeoStat_py · Bootstrap temporal para Google Colab

Este notebook prepara una sesión temporal de trabajo analítico para `GeoStat_py`.

> **Alcance:** motor de servicios/cálculo (sin UI desktop).


In [ ]:
# 1) Parámetros editables
REPO_URL = "https://github.com/<org>/<repo>.git"  # <- edita esta URL
BRANCH = ""  # opcional, ej: "main" o "develop"
BASE_DIR = "/content"
REPO_FOLDER_NAME = "GeoStat_py"
MOUNT_DRIVE = False
DRIVE_MOUNT_POINT = "/content/drive"
CSV_PATH_OPTIONAL = ""  # opcional, ej: "/content/drive/MyDrive/datos/muestras.csv"

REPO_DIR = f"{BASE_DIR.rstrip('/')}/{REPO_FOLDER_NAME}"
print("Parámetros cargados.")
print({
    "REPO_URL": REPO_URL,
    "BRANCH": BRANCH,
    "REPO_DIR": REPO_DIR,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "CSV_PATH_OPTIONAL": CSV_PATH_OPTIONAL,
})


In [ ]:
# 2) Validación visible de entorno (antes de bootstrap)
import os
import sys
from pathlib import Path

print("Python version:", sys.version.replace("\n", " "))
print("Python executable:", sys.executable)
print("Current working directory:", Path.cwd())
print("Repo dir objetivo:", REPO_DIR)
print("Repo existe ya:", Path(REPO_DIR).exists())
print("sys.path[0] actual:", sys.path[0] if sys.path else "")


In [ ]:
# 3) Obtener módulo bootstrap.py (clona temporalmente si aún no existe el repo)
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    print("Repo no existe todavía. Clonando para acceder a colab/bootstrap.py ...")
    clone_cmd = ["git", "clone", REPO_URL, str(repo_dir)]
    if BRANCH.strip():
        clone_cmd = ["git", "clone", "--branch", BRANCH.strip(), REPO_URL, str(repo_dir)]
    result = subprocess.run(clone_cmd, text=True, capture_output=True, check=False)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"No se pudo clonar el repo inicialmente (code={result.returncode}).")
else:
    print("Repo ya existe; se usará para importar bootstrap.py")

bootstrap_module_path = str(repo_dir / "colab")
if bootstrap_module_path in sys.path:
    sys.path.remove(bootstrap_module_path)
sys.path.insert(0, bootstrap_module_path)

from bootstrap import (
    clone_or_update_repo,
    configure_sys_path,
    create_service,
    environment_snapshot,
    install_requirements,
    mount_drive_if_requested,
    optional_csv_smoke_check,
    validate_imports,
)
print("bootstrap.py importado correctamente desde:", bootstrap_module_path)


In [ ]:
# 4) Bootstrap real: Drive opcional + clone/pull + instalación + sys.path
from pathlib import Path

_ = mount_drive_if_requested(MOUNT_DRIVE, DRIVE_MOUNT_POINT)
repo_root = clone_or_update_repo(REPO_URL, REPO_DIR, BRANCH)
install_requirements(repo_root / "colab" / "requirements_colab.txt")
configure_sys_path(repo_root)

print("\nEstado de entorno luego del bootstrap:")
for k, v in environment_snapshot(repo_root).items():
    print(f"- {k}: {v}")


In [ ]:
# 5) Validación de imports reales del motor (y ruta de origen)
modules_to_validate = [
    "app.services.geostat_service",
    "app.services.visualization_service",
    "app.services.variography_application_service",
    "app.models.dataset_model",
    "app.models.workflow_state_model",
]

results = validate_imports(modules_to_validate)
all_ok = True
for item in results:
    status = "OK" if item.ok else "ERROR"
    print(f"[{status}] {item.module_name}")
    if item.ok:
        print("   origin:", item.origin)
    else:
        print("   error:", item.error)
        all_ok = False

if not all_ok:
    raise RuntimeError("Falló validación de imports del motor.")


In [ ]:
# 6) Smoke checks del motor analítico (sin arrancar UI)
service = create_service()
print("GeostatService creado:", type(service).__name__)
print("Workflow status sample:", service.get_workflow_step_status()[:2])

smoke = optional_csv_smoke_check(service, CSV_PATH_OPTIONAL)
print("\nResultado smoke CSV opcional:")
for k, v in smoke.items():
    print(f"- {k}: {v}")

if smoke.get("executed") and not smoke.get("ok"):
    print("\n[WARN] El smoke CSV falló. Revisa CSV_PATH_OPTIONAL o formato de columnas.")
else:
    print("\nEntorno listo para trabajo analítico temporal en Colab.")


## Notas
- Este notebook **no** ejecuta la UI desktop.
- Todo queda aislado en la capa `colab/` y es reversible.
- Puedes continuar en nuevas celdas usando `service` para cargas, EDA, swath y variografía.
